# Battery-aware Colab Runner (Drive-persistent, stable)

배터리 상태(`BatteryShapeFormationEnv`) 학습/평가용 노트북.

**런타임 끊김 대비**: 모든 체크포인트/TB 로그/GIF/eval 텍스트를 처음부터 Google Drive에 직접 씀. 런타임이 죽어도 다음 세션에서 Drive를 마운트하면 그대로 이어 학습 가능.

**안정 셋업**: PPO 하이퍼파라미터를 안정 학습용으로 통일 (lr 2e-4 / ent 0.01 / clip 0.15 / batch 8192 / mb 512 / epochs 6 / completion 50).

런타임: `런타임 → 런타임 유형 변경 → GPU (T4)` 권장.

## 1. 기존 폴더 제거 후 GitHub clone

`Saehoon` 브랜치에 최신 배터리 코드가 있음.

In [ ]:
BRANCH = "Saehoon"

!rm -rf RL-2026s1-tp
!git clone https://github.com/umbrellalily/RL-2026s1-tp.git
%cd RL-2026s1-tp

!git fetch origin
!git switch $BRANCH
!git pull origin $BRANCH

# 받은 코드의 fix 여부 확인 (c0dda66 이후여야 함)
!git log --oneline -1
!grep -n 'infos\[agent\]\["battery"\]' comm_env.py || echo "  OK: 버그 라인 없음"

## 2. Google Drive 마운트 + 저장 경로

**모든 산출물은 여기 직결**. 런타임 죽어도 살아남는 위치.

```
/content/drive/MyDrive/drone_results/battery/
  ├ ckpts/   ← .pt 체크포인트
  ├ runs/    ← TensorBoard 로그
  ├ gifs/    ← 평가 GIF
  └ evals/   ← 평가 텍스트
```

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/drone_results/battery"
!mkdir -p {DRIVE_ROOT}/ckpts {DRIVE_ROOT}/runs {DRIVE_ROOT}/gifs {DRIVE_ROOT}/evals
print("DRIVE_ROOT =", DRIVE_ROOT)
!ls -la {DRIVE_ROOT}

## 3. 패키지 설치

In [ ]:
!pip install -q torchrl pettingzoo==1.24.3 gymnasium scipy matplotlib pillow tensorboard

## 4. Smoke test

`BatteryShapeFormationEnv` 로딩 + 한 스텝 돌려서 배터리 감소 확인.

In [ ]:
%cd /content/RL-2026s1-tp

import comm_env
from comm_env import BatteryShapeFormationEnv, ShapeFormationEnv

env = BatteryShapeFormationEnv(grid_size=25, n_agents=14, max_steps=10)
base = ShapeFormationEnv(grid_size=25, n_agents=14, max_steps=10)
print("baseline obs_dim:", base.obs_dim)
print("battery  obs_dim:", env.obs_dim, "(expected", base.obs_dim + 1 + (14 - 1), ")")

obs, infos = env.reset(seed=0)
print("reset infos[drone_0] keys:", list(infos.get('drone_0', {}).keys()), "(빈 리스트여야 함)")
print("all batteries start at 1.0:", set(round(env.battery[a], 5) for a in env.possible_agents))

acts = {a: (0 if i % 2 == 0 else 4) for i, a in enumerate(env.possible_agents)}
obs2, rewards, term, trunc, infos = env.step(acts)
for i, a in enumerate(env.possible_agents[:6]):
    print(f"  {a:9s} act={acts[a]}  battery={env.battery[a]:.5f}  reward={rewards[a]:+.4f}")

## 5. 빠른 학습 테스트 (sanity check)

1-2분 안에 끝남. reward가 점진 상승하고 ckpt가 Drive에 떨어지는지만 확인.

안정 셋업의 축소판 (`total-frames 16384`, batch 8192 → 2 iters, ckpt-every 1 → 2 ckpts).

In [ ]:
%cd /content/RL-2026s1-tp
DRIVE_ROOT = "/content/drive/MyDrive/drone_results/battery"

!python comm_train_battery.py \
  --grid-size 25 --n-agents 14 --max-steps 120 \
  --shapes GROUND,A \
  --completion-reward 50.0 \
  --assigned-target-reward 0.1 --coverage-delta-reward 0.2 \
  --hover-penalty 0.05 --shaping-coef 0.3 \
  --initial-battery 1.0 --hover-battery-cost 0.002 --move-battery-cost 0.005 \
  --low-battery-move-penalty 0.15 \
  --total-frames 16384 \
  --frames-per-batch 8192 --minibatch-size 512 --ppo-epochs 6 \
  --lr 2e-4 --ent-coef 0.01 --clip-eps 0.15 \
  --ckpt-every 1 \
  --save-dir {DRIVE_ROOT}/ckpts/battery_fast \
  --tb-logdir {DRIVE_ROOT}/runs/battery_fast

!ls -la {DRIVE_ROOT}/ckpts/battery_fast/

## 6. 본 학습: GROUND → A (안정 셋업, Drive 직저장)

**안정 셋업 (이전 5e-4 / 0.03 / 0.2 / 4096 / 256 / 3 / 100 → 변경):**
- `lr 2e-4` (1/2.5↓): 큰 step으로 인한 오버슈팅 억제
- `ent-coef 0.01` (1/3↓): 탐험 노이즈 ↓
- `clip-eps 0.15` (↓): PPO 업데이트 보수화
- `frames-per-batch 8192` / `minibatch-size 512` / `ppo-epochs 6`: gradient noise ↓ + 데이터 활용도 ↑
- `completion-reward 50` (↓): +100 cliff가 advantage를 폭발시키던 문제 완화

**총 frame 1,048,576 / batch 8192 = 128 iter**. `ckpt-every 2` → 64개 ckpt가 Drive에 직접 저장됨. 런타임 끊겨도 안 날아감.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/drone_results/battery"

!python comm_train_battery.py \
  --grid-size 25 --n-agents 14 --max-steps 120 \
  --shapes GROUND,A \
  --completion-reward 50.0 \
  --assigned-target-reward 0.1 --coverage-delta-reward 0.3 \
  --hover-penalty 0.05 --shaping-coef 0.5 \
  --initial-battery 1.0 --hover-battery-cost 0.002 --move-battery-cost 0.005 \
  --low-battery-move-penalty 0.15 \
  --total-frames 1048576 \
  --frames-per-batch 8192 --minibatch-size 512 --ppo-epochs 6 \
  --lr 2e-4 --ent-coef 0.01 --clip-eps 0.15 \
  --ckpt-every 2 \
  --save-dir {DRIVE_ROOT}/ckpts/battery_ground_A \
  --tb-logdir {DRIVE_ROOT}/runs/battery_ground_A

## 7. (선택) 이어 학습: 같은 task의 stable-finetune

⚠️ **주의**: 이 셀은 **A를 더 잘 만드는 용도**다. shape이 그대로 `GROUND,A`라서 다른 글자(B,C,...)에는 영향이 없다. 일반화 모델이 목표면 이 셀은 건너뛰고 **셀 22 (multi-shape curriculum)** 로 가야 한다.

쓸 만한 경우: 셀 12 학습 곡선이 피크에서 출렁이는데 **그 A 모델 자체를 deterministic하게 안정시키고 싶을 때**. 더 보수적 셋업 (lr 5e-5, clip 0.05, ent 0.003)으로 분산을 깎는다.

`BEST_ITER`는 학습 로그에서 가장 좋은 success/reward를 보인 iter로 바꿀 것.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/drone_results/battery"
BEST_ITER = 120  # ← 실제 학습 로그에서 best iter 번호로 바꿀 것

!python comm_train_battery.py \
  --grid-size 25 --n-agents 14 --max-steps 150 \
  --shapes GROUND,A \
  --completion-reward 50.0 \
  --assigned-target-reward 0.1 --coverage-delta-reward 0.2 \
  --hover-penalty 0.03 --shaping-coef 0.3 \
  --initial-battery 1.0 --hover-battery-cost 0.002 --move-battery-cost 0.005 \
  --low-battery-move-penalty 0.2 \
  --load-ckpt {DRIVE_ROOT}/ckpts/battery_ground_A/ckpt_{BEST_ITER}.pt \
  --total-frames 262144 \
  --frames-per-batch 8192 --minibatch-size 512 --ppo-epochs 4 \
  --lr 5e-5 --ent-coef 0.003 --clip-eps 0.05 \
  --ckpt-every 2 \
  --save-dir {DRIVE_ROOT}/ckpts/battery_ground_A_resume \
  --tb-logdir {DRIVE_ROOT}/runs/battery_ground_A_resume

## 8. TensorBoard (Drive 직독)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/drone_results/battery/runs

## 9. 평가 + 배터리 표시 GIF

각 드론 위에 배터리 막대(녹→황→적)와 퍼센트가 표시됨. 결과 텍스트엔 평균/최소/표준편차도 같이 남음.

`--greedy`로 deterministic 평가 권장 (학습 중 stochastic success의 출렁임이 사라짐). `BEST_ITER`는 실제 best ckpt 번호로 바꿀 것.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/drone_results/battery"
BEST_ITER = 120  # ← 학습 로그에서 best 또는 마지막 iter

!python comm_eval_battery.py \
  --ckpt {DRIVE_ROOT}/ckpts/battery_ground_A/ckpt_{BEST_ITER}.pt \
  --grid-size 25 --n-agents 14 --max-steps 120 \
  --shapes GROUND,A \
  --completion-reward 50.0 \
  --initial-battery 1.0 --hover-battery-cost 0.002 --move-battery-cost 0.005 \
  --low-battery-move-penalty 0.15 \
  --greedy --n-episodes 100 \
  --save-gif {DRIVE_ROOT}/gifs/demo_battery_ground_A.gif \
  --out {DRIVE_ROOT}/evals/eval_battery_ground_A.txt

## 10. Drive 산출물 점검 (학습/평가 후)

런타임 끊겨도 이 셀로 Drive 상태만 확인하면 됨.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/drone_results/battery"
print("=== ckpts ===")
!ls -la {DRIVE_ROOT}/ckpts/ 2>/dev/null
print("\n=== 가장 최근 학습 디렉토리들의 ckpt 개수 ===")
!for d in {DRIVE_ROOT}/ckpts/*/; do echo "$d -> $(ls $d 2>/dev/null | wc -l) files"; done
print("\n=== gifs ===")
!ls -la {DRIVE_ROOT}/gifs/ 2>/dev/null
print("\n=== evals ===")
!ls -la {DRIVE_ROOT}/evals/ 2>/dev/null

## 11. 다단계 전환 학습 (multi-shape) — **일반화 모델 핵심**

여러 글자를 만들 줄 아는 모델을 학습한다. 두 가지 방법:

**(A) Curriculum 전이 — 추천**
- 셀 12에서 `GROUND,A` 단일 글자로 먼저 학습 (쉬워서 수렴 빠름)
- 그 ckpt를 `--load-ckpt`로 받아 `GROUND,A,B,C,D` 멀티로 확장
- 관측 차원이 shape 수와 무관해서 그대로 로드 가능
- 처음부터 멀티 학습보다 안정적·빠름

**(B) 처음부터 multi**
- 셀 12 건너뛰고 바로 여기로
- `CURRICULUM = False`로 두면 됨

아래 셀의 `CURRICULUM` 변수로 두 옵션 전환. `BEST_ITER`는 셀 12 학습에서 가장 좋았던 iter.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/drone_results/battery"

# === Curriculum 옵션 ===
CURRICULUM = True       # True: 셀 12 A-학습 ckpt에서 전이, False: 처음부터 multi
BEST_ITER  = 120        # CURRICULUM=True일 때 사용할 A-ckpt iter 번호
# =====================

import os
load_ckpt_path = f"{DRIVE_ROOT}/ckpts/battery_ground_A/ckpt_{BEST_ITER}.pt"
load_arg = f"--load-ckpt {load_ckpt_path}" if CURRICULUM else ""

if CURRICULUM:
    assert os.path.exists(load_ckpt_path), f"curriculum ckpt 없음: {load_ckpt_path}"
    print("Curriculum 전이 학습:", load_ckpt_path)
else:
    print("Multi-shape 처음부터 학습")

# Curriculum이면 LR을 살짝 낮춰서 (1e-4) 사전학습 망가뜨리지 않고 새 task 추가
lr = "1e-4" if CURRICULUM else "2e-4"
ent = "0.005" if CURRICULUM else "0.01"

!python comm_train_battery.py \
  --grid-size 25 --n-agents 14 --max-steps 200 \
  --shapes GROUND,A,B,C,D \
  --completion-reward 50.0 \
  --assigned-target-reward 0.1 --coverage-delta-reward 0.3 \
  --hover-penalty 0.05 --shaping-coef 0.5 \
  --initial-battery 1.0 --hover-battery-cost 0.002 --move-battery-cost 0.005 \
  --low-battery-move-penalty 0.15 \
  --total-frames 2097152 \
  --frames-per-batch 8192 --minibatch-size 512 --ppo-epochs 6 \
  --lr {lr} --ent-coef {ent} --clip-eps 0.15 \
  --ckpt-every 2 \
  {load_arg} \
  --save-dir {DRIVE_ROOT}/ckpts/battery_ABCD \
  --tb-logdir {DRIVE_ROOT}/runs/battery_ABCD

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/drone_results/battery"
BEST_ITER = 120  # ← 학습 로그에서 best 또는 마지막 iter

!python comm_eval_battery.py \
  --ckpt {DRIVE_ROOT}/ckpts/battery_ABCD/ckpt_{BEST_ITER}.pt \
  --grid-size 25 --n-agents 14 --max-steps 200 \
  --shapes GROUND,A,B,C,D \
  --completion-reward 50.0 \
  --initial-battery 1.0 --hover-battery-cost 0.002 --move-battery-cost 0.005 \
  --low-battery-move-penalty 0.15 \
  --greedy --n-episodes 50 \
  --save-gif {DRIVE_ROOT}/gifs/demo_battery_ABCD.gif \
  --out {DRIVE_ROOT}/evals/eval_battery_ABCD.txt

## 12. (비교) 배터리 미고려 baseline 학습

기존 `comm_train.py`(배터리 OFF)을 **같은 안정 셋업으로** 학습해서 배터리 인지 모델과 비교. completion-reward 같은 보상 스케일도 동일하게 50으로 맞춤 → 공정 비교.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/drone_results/battery"

!python comm_train.py \
  --grid-size 25 --n-agents 14 --max-steps 120 \
  --shapes GROUND,A \
  --completion-reward 50.0 \
  --assigned-target-reward 0.1 --coverage-delta-reward 0.3 \
  --hover-penalty 0.05 --shaping-coef 0.5 \
  --total-frames 1048576 \
  --frames-per-batch 8192 --minibatch-size 512 --ppo-epochs 6 \
  --lr 2e-4 --ent-coef 0.01 --clip-eps 0.15 \
  --ckpt-every 2 \
  --save-dir {DRIVE_ROOT}/ckpts/baseline_ground_A \
  --tb-logdir {DRIVE_ROOT}/runs/baseline_ground_A